# Davinci Notebook 17 — Long-Context Curriculum & Synthetic Retrieval Tasks

Planning and evaluation utilities for gradually expanding context. Generates controlled synthetic retrieval examples and curriculum manifests; does not constitute long-context training or prove model capability. Synthetic tasks are for sanity checks and must not be the sole benchmark.

In [ ]:
import random, json, hashlib, math
from pathlib import Path
random.seed(2026)


## 1. Context-length curriculum
Progressively expand sequence length only after baseline quality and memory/throughput checks. Suggested stages are a conservative experiment plan, not guarantees.

In [ ]:
CURRICULUM=[{'tokens':4096,'phase':'base','gate':'stable loss and baseline evaluation'}, {'tokens':8192,'phase':'extend','gate':'short-context regression check'}, {'tokens':16384,'phase':'extend','gate':'memory and quality checks'}, {'tokens':32768,'phase':'extend','gate':'long-context retrieval evaluation'}, {'tokens':65536,'phase':'extend','gate':'position and cache validation'}, {'tokens':131072,'phase':'extend','gate':'kernel and throughput validation'}, {'tokens':262144,'phase':'extend','gate':'memory, quality and latency review'}, {'tokens':500000,'phase':'experimental','gate':'research-grade kernel, quality, memory and retrieval validation'}]
print(json.dumps(CURRICULUM,indent=2))


## 2. Synthetic needle retrieval example generator
Produces examples with a unique key/value fact inserted in a distractor document. Use tokenized lengths to measure real context sizes; character count is not token count.

In [ ]:
def make_needle_example(target_id,needle_position,documents=20,seed=42):
 rng=random.Random(seed); key=f'fact_key_{target_id:06d}'; value=f'fact_value_{rng.randrange(10**8):08d}'
 chunks=[f'Distractor document {i}: {rng.choice(["The market report describes regional trends.","A project summary discusses implementation details.","The note records a meeting and action items."])}' for i in range(documents)]
 needle=f'Record: {key} is associated with {value}.'; pos=max(0,min(documents,needle_position)); chunks.insert(pos,needle)
 return {'prompt':'\n\n'.join(chunks)+'\n\nQuestion: What value is associated with '+key+'?','answer':value,'needle_index':pos,'metadata':{'synthetic':True,'seed':seed}}
ex=make_needle_example(3,10); print(ex['answer'],ex['needle_index'])


## 3. Build a stratified synthetic test set
Position and distractor count vary; persist seed and metadata for reproducibility.

In [ ]:
def build_retrieval_test(n=100,seed=123):
 rng=random.Random(seed); out=[]
 for i in range(n):
  pos=rng.randrange(0,40); docs=rng.randrange(10,60); out.append(make_needle_example(i,pos,docs,seed+i))
 return out
testset=build_retrieval_test(100); print('examples:',len(testset))


## 4. Scoring and manifest helpers

In [ ]:
def exact_match(pred,answer): return str(pred).strip()==str(answer).strip()
def score_predictions(predictions,examples):
 assert len(predictions)==len(examples); scores=[exact_match(p,e['answer']) for p,e in zip(predictions,examples)]; return {'n':len(scores),'accuracy':sum(scores)/max(1,len(scores))}
def write_testset_jsonl(items,path):
 Path(path).write_text(''.join(json.dumps(x,ensure_ascii=False)+'\\n' for x in items),encoding='utf-8')
 manifest={'count':len(testset),'seed':123,'task':'synthetic needle retrieval','synthetic_only':True,'warning':'not a substitute for real long-context evaluation'}; manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest(); print(manifest)


## 5. Long-context experiment gates

- Report exact token length, not character estimates.
- Measure evidence at beginning, middle, and end.
- Add distractors, multiple needles, cross-document reasoning, and retrieval failure cases.
- Compare against short-context baseline and shuffled evidence controls.
- Record KV-cache memory and prefill latency separately from decode latency.
- Verify model does not answer from memorized benchmark content.
- Do not claim 500K useful context from a configuration or positional extrapolation alone.